
# Ordenamiento Automático de Imágenes con Inteligencia Artificial

Este notebook documenta completamente el sistema de clasificación automática de imágenes 
utilizando un modelo previamente entrenado en TensorFlow/Keras.

## Objetivo

Automatizar el proceso de:
- Búsqueda recursiva de imágenes dentro de una carpeta.
- Clasificación mediante un modelo de Deep Learning.
- Filtrado por nivel mínimo de confianza.
- Organización automática en carpetas según la clase predicha.
- Renombrado del archivo agregando el porcentaje de confianza.


## 1. Importaciones

En este apartado se incorporan todas las librerías necesarias para el funcionamiento del sistema de clasificación automática.

Se utiliza TensorFlow/Keras para la carga y ejecución del modelo de Deep Learning previamente entrenado. La biblioteca NumPy permite el manejo eficiente de arreglos numéricos durante el procesamiento de imágenes.

El módulo tensorflow.keras.preprocessing.image se emplea para cargar y transformar las imágenes al formato requerido por la red neuronal. Asimismo, ImageDataGenerator permite reconstruir las clases originales del dataset para mantener coherencia con el entrenamiento.

Por otro lado, argparse habilita la ejecución del sistema desde línea de comandos, permitiendo recibir dinámicamente la ruta de entrada. Finalmente, pathlib y shutil se utilizan para la gestión estructurada de rutas y la copia organizada de archivos clasificados.

Este conjunto de importaciones establece la base funcional del sistema.

In [1]:

import argparse
import shutil
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from pathlib import Path


2026-02-24 12:50:40.958950: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-24 12:50:40.963231: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-24 12:50:40.974873: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-24 12:50:40.992552: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-24 12:50:40.997509: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-24 12:50:41.012094: I tensorflow/core/platform/cpu_feature_gu

## 2. Configuración de Rutas

En esta sección se definen las rutas principales del proyecto utilizando objetos Path, lo que permite una gestión multiplataforma y evita dependencias de rutas absolutas escritas manualmente.

Se establece como referencia el directorio base del proyecto y, a partir de este, se construyen dinámicamente las rutas hacia:

- El modelo entrenado (models)
- El dataset original de entrenamiento
- El directorio de salida donde se almacenarán las imágenes clasificadas

Este enfoque mejora la portabilidad del sistema, facilita su despliegue en distintos entornos y garantiza una estructura organizada del proyecto. Además, evita errores derivados de rutas codificadas directamente como cadenas de texto.

In [2]:

BASE_DIR = Path.cwd()
PROYECT_DIR = BASE_DIR.parent
MODELS_DIR = (PROYECT_DIR / "models").resolve()
DATA_DIR = (PROYECT_DIR / "data" / "raw").resolve()

MODEL_PATH = (MODELS_DIR / "modelo_de_filtrado.keras").resolve()
DATASET_PATH = (DATA_DIR / "waste_classification" / "images" / "dataset_final").resolve()
OUTPUT_DIR = (DATA_DIR / "clasificated").resolve()


## 3. Configuración Global

En este apartado se definen los parámetros fundamentales que controlan el comportamiento del sistema de clasificación.

Se establece el tamaño estándar de entrada de las imágenes (IMG_SIZE), el cual debe coincidir con el utilizado durante el entrenamiento del modelo. También se define el tamaño de lote (BATCH_SIZE) empleado por el generador de datos.

Se especifican las extensiones de archivo válidas para asegurar que únicamente se procesen formatos de imagen compatibles.

Finalmente, se define el umbral mínimo de confianza (MIN_CONFIDENCE = 0.60), el cual determina si una predicción será aceptada o descartada. Este parámetro permite filtrar clasificaciones poco confiables, aumentando la precisión práctica del sistema en entornos reales.

In [3]:

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
MIN_CONFIDENCE = 0.60


## 4. Carga del Modelo

En esta sección se implementa la función encargada de cargar el modelo de red neuronal previamente entrenado y almacenado en formato .keras.

El modelo es recuperado desde el sistema de archivos mediante tf.keras.models.load_model, lo que permite restaurar tanto la arquitectura como los pesos aprendidos durante el entrenamiento.

Este proceso es fundamental, ya que el modelo constituye el núcleo del sistema de clasificación. Sin esta carga inicial, no sería posible realizar inferencias sobre nuevas imágenes.

Al centralizar la carga del modelo en una función específica, se mejora la modularidad del código y se facilita su reutilización en otros entornos o aplicaciones.

In [4]:

def load_trained_model(model_path: Path):
    model = tf.keras.models.load_model(model_path)
    return model


## 5. Obtención de Clases

Este apartado tiene como propósito reconstruir la correspondencia entre los índices numéricos generados por el modelo y los nombres reales de las clases.

Durante el entrenamiento, el generador de datos asigna automáticamente un índice entero a cada categoría presente en el dataset. Para garantizar coherencia durante la fase de inferencia, se utiliza nuevamente ImageDataGenerator.flow_from_directory para recuperar la estructura original de clases.

Posteriormente, se crea un diccionario inverso (index_to_class) que permite traducir el índice con mayor probabilidad predicho por el modelo al nombre real de la clase correspondiente.

Este procedimiento asegura que las predicciones realizadas en producción mantengan consistencia con el entrenamiento original.

In [5]:

def load_class_indices(dataset_path: Path):
    datagen = ImageDataGenerator(rescale=1.0 / 255)

    generator = datagen.flow_from_directory(
        str(dataset_path),
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        shuffle=False,
    )

    class_indices = generator.class_indices
    index_to_class = {v: k for k, v in class_indices.items()}

    return index_to_class


## 6. Preprocesamiento de Imagen

En esta sección se define el procedimiento que transforma cada imagen antes de ser enviada al modelo para su clasificación.

El proceso incluye los siguientes pasos:

- Carga de la imagen desde el sistema de archivos.
- Redimensionamiento al tamaño requerido por la red neuronal.
- Conversión a un arreglo NumPy.
- Normalización de los valores de píxeles al rango [0, 1].
- Expansión de dimensiones para agregar el eje correspondiente al batch.

Este preprocesamiento es esencial, ya que los modelos de Deep Learning requieren que las entradas mantengan el mismo formato y escala utilizados durante el entrenamiento.

Una transformación inconsistente podría generar predicciones incorrectas o inestables.

In [6]:

def preprocess_image(img_path: Path):
    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img)
    img_array = img_array / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    return img_array


## 7. Predicción

En este apartado se define la función encargada de realizar la inferencia sobre una imagen individual utilizando el modelo previamente cargado.

El procedimiento consiste en enviar la imagen preprocesada al modelo mediante el método predict, el cual devuelve un arreglo de probabilidades correspondientes a cada clase aprendida durante el entrenamiento.

Posteriormente:

- Se identifica el índice con mayor probabilidad utilizando argmax.
- Se obtiene el valor máximo como nivel de confianza.
- Se traduce el índice numérico al nombre real de la clase mediante el diccionario index_to_class.

El resultado final de esta función es una tupla que contiene la clase predicha y el nivel de confianza asociado, lo que permite aplicar posteriormente un criterio de aceptación o descarte.

In [7]:

def predict_image(model, img_path: Path, index_to_class: dict):
    img_array = preprocess_image(img_path)
    predictions = model.predict(img_array, verbose=0)

    predicted_index = np.argmax(predictions[0])
    confidence = np.max(predictions[0])
    predicted_class = index_to_class[predicted_index]

    return predicted_class, confidence


## 8. Búsqueda Recursiva

Este apartado implementa la lógica necesaria para localizar todas las imágenes válidas dentro de una carpeta y sus subcarpetas.

Se utiliza el método rglob("*") de pathlib, que permite recorrer recursivamente toda la estructura de directorios. Posteriormente, se filtran los archivos según:

- Que sean archivos reales (no carpetas).
- Que su extensión coincida con los formatos permitidos.

Este enfoque permite que el sistema procese grandes volúmenes de imágenes organizadas en múltiples niveles de directorios, facilitando su uso en escenarios reales donde los archivos no siempre están estructurados de manera uniforme.

In [8]:

def find_images_in_directory(directory: Path):
    return [
        path for path in directory.rglob("*")
        if path.suffix.lower() in VALID_EXTENSIONS and path.is_file()
    ]


## 9. Guardado de Imagen Clasificada

En esta sección se define el procedimiento para almacenar las imágenes clasificadas de manera organizada.

Si la predicción cumple con el umbral mínimo de confianza:

- Se crea automáticamente una carpeta correspondiente a la clase predicha (si no existe).
- Se renombra el archivo agregando al inicio el porcentaje entero de confianza.
- Se verifica que no exista otro archivo con el mismo nombre para evitar sobreescritura.
- Se copia la imagen al nuevo destino.

El renombrado con el porcentaje permite visualizar rápidamente el nivel de seguridad de la clasificación sin necesidad de revisar registros adicionales.

Este diseño mejora la trazabilidad y facilita auditorías posteriores del sistema.

In [9]:

def save_classified_image(img_path: Path, predicted_class: str, confidence: float):
    class_folder = OUTPUT_DIR / predicted_class
    class_folder.mkdir(parents=True, exist_ok=True)

    percentage = int(confidence * 100)
    new_filename = f"{percentage}_{img_path.name}"
    destination = class_folder / new_filename

    counter = 1
    while destination.exists():
        new_filename = f"{percentage}_{img_path.stem}_{counter}{img_path.suffix}"
        destination = class_folder / new_filename
        counter += 1

    shutil.copy2(img_path, destination)


## 10. Clasificación Completa

Este apartado integra todas las funciones anteriores en un flujo de ejecución coherente.

El proceso general es:

- Buscar todas las imágenes dentro del directorio indicado.
- Procesarlas una por una.
- Obtener la predicción y el nivel de confianza.
- Comparar la confianza con el umbral definido.
- Guardar la imagen si cumple el criterio o descartarla si no lo cumple.
- Manejar posibles errores sin detener la ejecución completa.

Este diseño permite que el sistema continúe operando incluso si una imagen presenta problemas de lectura o formato, lo que mejora su robustez en entornos reales.

En conjunto, esta función representa la implementación operativa del sistema de ordenamiento automático basado en inteligencia artificial.

In [10]:
def classify_directory(model, index_to_class, input_path: Path):
    images = find_images_in_directory(input_path)

    if not images:
        print("No se encontraron imágenes.")
        return

    print(f"Se encontraron {len(images)} imágenes.\n")

    for img_path in images:
        try:
            predicted_class, confidence = predict_image(
                model, img_path, index_to_class
            )

            if confidence >= MIN_CONFIDENCE:
                save_classified_image(img_path, predicted_class, confidence)
                print(f"{img_path.name} → {predicted_class} ({confidence*100:.2f}%)")
            else:
                print(f"{img_path.name} descartada ({confidence*100:.2f}%)")

        except Exception as e:
            print(f"Error con {img_path.name}: {e}")

## 11. Solicitud de Carpeta al Usuario

En este apartado se adapta el sistema para que funcione completamente dentro del notebook, eliminando la dependencia de argumentos por línea de comandos.

Se solicita al usuario que ingrese manualmente la ruta de la carpeta que contiene las imágenes a clasificar. Posteriormente:

- Se valida que la ruta exista y sea un directorio.
- Se crea el directorio de salida si no existe.
- Se cargan el modelo y las clases.
- Se ejecuta el proceso completo de clasificación.

Este diseño es adecuado para entornos académicos y pruebas controladas, donde la ejecución interactiva facilita la experimentación.

In [11]:
# Solicitar ruta al usuario
input_path_str = input("Ingrese la ruta de la carpeta que contiene las imágenes: ")
input_path = Path(input_path_str).resolve()

# Validar ruta
if not input_path.exists() or not input_path.is_dir():
    print("La ruta proporcionada no existe o no es válida.")
else:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print("Cargando modelo...")
    model = load_trained_model(MODEL_PATH)

    print("Cargando clases...")
    index_to_class = load_class_indices(DATASET_PATH)

    print("Iniciando clasificación...\n")
    classify_directory(model, index_to_class, input_path)

    print("\nProceso finalizado.")

Cargando modelo...


I0000 00:00:1771959050.332837  427653 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-02-24 12:50:50.335841: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2343] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Cargando clases...
Found 6147 images belonging to 4 classes.
Iniciando clasificación...

Se encontraron 15000 imágenes.

Image_46.png → metal (64.49%)
Image_79.png → plastic (80.02%)
Image_70.png → metal (99.03%)
Image_84.png → metal (70.01%)
Image_81.png → metal (99.03%)
Image_176.png descartada (59.97%)
Image_28.png → metal (83.64%)
Image_56.png → metal (99.95%)
Image_183.png → metal (84.17%)
Image_72.png → metal (82.21%)
Image_33.png → metal (99.87%)
Image_215.png → metal (99.88%)
Image_151.png → plastic (66.41%)
Image_146.png → metal (99.97%)
Image_250.png → plastic (89.66%)
Image_212.png → plastic (66.18%)
Image_9.png → metal (93.61%)
Image_230.png → metal (75.67%)
Image_21.png → metal (76.45%)
Image_52.png descartada (59.15%)
Image_43.png → metal (61.16%)
Image_1.png → metal (97.59%)
Image_100.png → metal (97.70%)
Image_44.png → metal (100.00%)
Image_105.png → metal (99.97%)
Image_147.png → metal (82.21%)
Image_77.png descartada (56.91%)
Image_223.png → metal (87.70%)
Image_114.p